# 02 · TF-IDF, E5 a cross-encoder v reálném pipeline

Tyto notebooky používají skutečné soubory projektu, ne vymyšlené výsledky. Buňky bez dat skončí vysvětlením, které balíčky chybí. Klon repozitáře může vyžadovat přístup; alternativně nahraj jeho kopii do Colab. API klíč pro tuto analýzu nepotřebuješ. Trénování ani nové vyhledávání se bez přepínače nespouští. Rezervované testové otázky se nečtou ani nevyhodnocují.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, hashlib
# In Colab clone the project; locally reuse the current checkout.
if Path('src/librarian').is_dir():
    ROOT = Path.cwd()
elif Path('ML_Librarian/src/librarian').is_dir():
    ROOT = Path('ML_Librarian').resolve()
else:
    subprocess.run(['git','clone','https://github.com/Kilitar/ML_Librarian.git'],check=True)
    ROOT = Path('ML_Librarian').resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src/librarian'))
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
# Set an immutable git ref before starting for exact code-version reproduction.


In [ ]:
# Optional environment setup. Enable in a fresh Colab runtime; CPU is sufficient.
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable,'-m','pip','install','-r','requirements-librarian.txt'],check=True)


In [ ]:
# Operator supplies the ignored data bundle(s), never an API key.
UPLOAD_BUNDLES = False  # In Colab set True and select runtime.local.zip and/or development.local.zip.
if UPLOAD_BUNDLES:
    from google.colab import files
    uploaded = files.upload()
    import io, zipfile
    expected = json.loads((ROOT/'deployment/artifacts.json').read_text())
    for filename, blob in uploaded.items():
        record = expected.get(filename)
        if record is None or hashlib.sha256(blob).hexdigest() != record['sha256']:
            raise ValueError('Unknown/stale bundle: compare with operator manifest; do not bypass SHA check')
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            names=z.namelist()
            manifest=json.loads(z.read('bundle_manifest.json'))
            if len(names)!=len(set(names)) or set(names)!=set(manifest)|{'bundle_manifest.json'}:
                raise ValueError('Invalid bundle manifest')
            if sum(i.file_size for i in z.infolist())>40*1024*1024:raise ValueError('Bundle too large')
            staged=[]
            for name, digest in manifest.items():
                if name not in record['files']:raise ValueError('Unexpected member')
                target=(ROOT/name).resolve()
                if not target.is_relative_to(ROOT.resolve()):raise ValueError('Unsafe path')
                content=z.read(name)
                if hashlib.sha256(content).hexdigest()!=digest:raise ValueError('Member SHA mismatch')
                staged.append((target,content))
            for target,content in staged:
                target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(content)
        print('Verified bundle:',filename)


## Lexikální baseline na skutečném korpusu
Bez nové inferenční služby; vypíšeme i skutečný text a chunk ID.

In [ ]:
path=ROOT/'data/normalized/chunks_curated.jsonl'
if not path.exists():
    print('Chybí runtime.local.zip.')
else:
    from sklearn.feature_extraction.text import TfidfVectorizer
    import numpy as np
    rows=[json.loads(s) for s in path.read_text(encoding='utf-8').splitlines()]
    query='Proč se před k-NN škálují příznaky?'
    vectorizer=TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=1)
    matrix=vectorizer.fit_transform([r['text'] for r in rows])
    scores=(matrix @ vectorizer.transform([query]).T).toarray().ravel()
    for i in np.argsort(-scores)[:5]:
        print(float(scores[i]),rows[i]['source_path'],rows[i]['chunk_id'])
        print(rows[i]['text'])


## Volitelné skutečné E5/rerank vyhledání
Stáhne připnuté veřejné váhy, může zabrat RAM/čas. Není to nové hodnocení accuracy. Známá ukázková otázka je development příklad.

In [ ]:
RUN_NEURAL_SEARCH = False
if RUN_NEURAL_SEARCH:
    from prepare_runtime_models import prepare
    prepare()
    from search import E5SearchIndex
    index=E5SearchIndex.load()
    for mode in ('e5','hybrid','rerank'):
        print('MODE:',mode)
        for row in index.search('Why should features be scaled before k-NN?',mode=mode):
            print(row['chunk_id'],row.get('ranking_score',row['score']))
            print(row['text'])
else:
    print('Neural search vypnuto; pro spuštění nastav RUN_NEURAL_SEARCH=True.')
